<a href="https://colab.research.google.com/github/mafloan/Agente-IA-BimBam-Buy/blob/main/Agente_IA_BB_Buy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sys
!{sys.executable} -m pip install langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader

FILES = [
    ("/mnt/data/Guia de tiempos y costos.pdf", "envios"),
    ("/mnt/data/Manual de Garantía.pdf", "garantia"),
    ("/mnt/data/Politica de reembolsos.pdf", "reembolsos"),
    ("/mnt/data/Preguntas frecuentes.pdf", "faq"),
    ("/mnt/data/Programa de afiliados.pdf", "afiliados"),
]

def load_documents():
    all_docs = []

    for path, doc_type in FILES:
        loader = PyPDFLoader(path)
        docs = loader.load()

        for d in docs:
            d.metadata["source_doc"] = doc_type  # 🔥 KEY
            d.metadata["file_name"] = path.split("/")[-1]

        all_docs.extend(docs)

    return all_docs

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "]  # better for Spanish structure
    )
    return splitter.split_documents(documents)

In [6]:
from langchain_community.embeddings import CohereEmbeddings

def create_embeddings(api_key):
    return CohereEmbeddings(
        cohere_api_key=api_key,
        model="embed-multilingual-v3.0",  # 🔥 IMPORTANT CHANGE
        user_agent="langchain"
    )

In [8]:
!pip install cohere
embeddings = create_embeddings(COHERE_API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 67.2 MB/s eta 0:00:00


KeyError: 'user_agent'

In [ ]:
# Used to securely store your API key
from google.colab import userdata

COHERE_API_KEY=userdata.get('COHERE_API_KEY')

In [ ]:
from langchain_community.vectorstores import FAISS

def create_vectorstore(chunks, embeddings):
    return FAISS.from_documents(chunks, embeddings)

In [ ]:
def create_retriever(vectorstore):
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(retriever, google_api_key):

    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-pro",
        google_api_key=google_api_key,
        temperature=0.2
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         """Eres un asistente experto en políticas de BimBam Buy.

Usa SOLO la información del contexto.

Reglas:
- Responde en español
- Si no sabes la respuesta, di: "No tengo información suficiente"
- Sé claro y directo
- Si aplica, menciona condiciones o excepciones

Contexto:
{context}"""),
        ("human", "{question}")
    ])

    rag_chain = (
        RunnableParallel({
            "docs": lambda x: retriever.get_relevant_documents(x["question"]),
            "question": RunnablePassthrough()
        })
        | RunnableParallel({
            "answer": (
                lambda x: {
                    "context": "\n\n".join([d.page_content for d in x["docs"]]),
                    "question": x["question"]
                }
                | prompt
                | llm
                | StrOutputParser()
            ),
            "sources": lambda x: [
                {
                    "doc": d.metadata["source_doc"],
                    "page": d.metadata.get("page", None),
                    "file": d.metadata["file_name"]
                }
                for d in x["docs"]
            ]
        })
    )

    return rag_chain

In [ ]:
docs = load_documents()
chunks = split_documents(docs)

embeddings = create_embeddings(COHERE_API_KEY)
vectorstore = create_vectorstore(chunks, embeddings)

retriever = create_retriever(vectorstore)
rag_chain = create_rag_chain(retriever, GOOGLE_API_KEY)

query = "¿Cuándo aplica un reembolso?"

result = rag_chain.invoke({"question": query})

print(result["answer"])
print(result["sources"])

In [ ]:
vectorstore.save_local("faiss_index")

In [ ]:
retriever.get_relevant_documents(
    "reembolso",
    filter={"source_doc": "reembolsos"}
)